# Ablation: Training Data Size

Compares **1k vs 2.5k vs 5k training samples** for SigExt,
holding base model and threshold fixed.

Symmetrically applied to both IT and EN.

**Memory strategy**: SigExt on CPU -> unloaded -> single LLM per language.

In [ ]:
import warnings, os
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!uv pip install -e ../..
load_dotenv()
from huggingface_hub import login
login(token=os.getenv('HF_TOKEN'))


In [ ]:
from sm_sip.config import SigExtConfig
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, unload_sigext_model, load_llm, create_summary_chain, preprocess_dataset
from sm_sip.prompts import get_summary_prompt
from sm_sip.pipelines import run_inference, run_evaluation
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory

## Configuration

Fix threshold at 0.60, vary sample size (1k, 2.5k, 5k).

In [ ]:
THRESHOLD = '060t'
SIZES = ['1k', '2500', '5k']

ABLATION_PRESETS = {
    'it': [f'xlmr-{s}-{THRESHOLD}' for s in SIZES],
    'en': [f'xlmr-{s}-{THRESHOLD}' for s in SIZES],
}

NUM_TEST_SAMPLES = 50

print('Presets to evaluate:')
for lang, presets in ABLATION_PRESETS.items():
    for p in presets:
        print(f'  {lang}: {p}')

## Evaluation Loop

In [ ]:
results = {}

for lang, presets in ABLATION_PRESETS.items():
    print(f'\n{"="*70}')
    print(f'  Language: {lang.upper()}')
    print(f'{"="*70}')

    data = get_test_data(lang=lang, num_samples=NUM_TEST_SAMPLES,
                         skip_samples=SigExtConfig.LANG_DATASETS[lang]['skip'])

    _, _, pipe = load_llm('meta-llama/Llama-3.1-8B-Instruct', '8bit')
    chain = create_summary_chain(pipe, get_summary_prompt(lang, 'source_aware'))

    for preset in presets:
        key = f'{lang}_{preset}'
        print(f'\n  -> {key}')

        try:
            sc = SigExtConfig.from_preset(lang, preset)
            sm, st = load_sigext_model(sc.model_id, device='cpu')
            proc = preprocess_dataset(data, sm, st, lang=lang)
            unload_sigext_model(sm, st)

            res = run_inference(proc, chain)
            metrics = run_evaluation(res, lang=lang)
            results[key] = metrics
            print(f'    BERTScore: {metrics["bert_score"]["mean"]:.4f}  '
                  f'ROUGE-1: {metrics["rouge1"]["mean"]:.4f}  '
                  f'KIR: {metrics["kir"]["mean"]:.4f}')
        except Exception as e:
            print(f'    FAILED: {e}')
            results[key] = {'error': str(e)}

    clear_gpu_memory()

## Results Table

In [ ]:
print('\n' + '='*85)
print('  TRAINING DATA SIZE COMPARISON (1k vs 2.5k vs 5k)')
print('='*85)
print(f'{"Config":<30} {"BERTScore":>10} {"ROUGE-1":>10} {"ROUGE-L":>10} {"KIR":>10}')
print('-'*75)

for key in sorted(results.keys()):
    m = results[key]
    if 'error' in m:
        print(f'{key:<30} {"ERROR":>10}')
        continue
    print(f'{key:<30} '
          f'{m["bert_score"]["mean"]:>10.4f} '
          f'{m["rouge1"]["mean"]:>10.4f} '
          f'{m["rougeL"]["mean"]:>10.4f} '
          f'{m["kir"]["mean"]:>10.4f}')

print('-'*75)
print()
print('Key question: Does more training data (1k -> 2.5k -> 5k) consistently')
print('improve downstream summarization quality for both languages?')

In [ ]:
save_results(
    {'ablation': 'training_data', 'results': results},
    'results/ablation_training_data.json',
)
print('Results saved to results/ablation_training_data.json')